# DWS-Bench: OLMo 2 1B No-CoT

Runs direct answer (no chain-of-thought) at three generation budgets (128, 256, 512 new tokens), each as an independent
`run_eval.py` process on the full benchmark (`data/full_benchmark.jsonl`).

No-CoT answers are ~10 tokens, so the three budgets normally give (near-)identical results; `run_eval.py` logs a warning about this and the runs are kept on purpose as independent replicates.

**Kaggle setup:** Accelerator = GPU (T4 ×2 or P100), Internet = On, and the StateMachine repository attached
as a Dataset (or set `REPO_URL`). Then *Run All*. If the repository has no `data/` folder the notebook
generates it with `generate_all.py --write`.

Outputs land in `/kaggle/working/<OUTPUT_NAME>/<model>_<mode>_<tokens>/<model>/<mode>_<tokens>/full_benchmark_*`
and are zipped to `/kaggle/working/<OUTPUT_NAME>.zip` (refreshed after every budget).
Set `MOCK = True` in the first cell for a 1-minute smoke test with the mock engine (no GPU, no downloads).
A budget that would not fit into the time guard (`SESSION_LIMIT_HOURS`) is skipped and reported, never silently dropped.


In [ ]:
# ---- Configuration (the only cell you should need to edit) -------------------
import time
NB_START = time.time()

MODEL = 'olmo-2-1b'                 # key in eval/models.py
USE_COT = False                  # True -> run_eval.py --cot
TOKEN_BUDGETS = [128, 256, 512]       # one independent run_eval.py call per budget
REQUIRES_HF_TOKEN = False          # gated repo -> Kaggle secret HF_TOKEN is mandatory
OUTPUT_NAME = 'dws_olmo1b_no_cot'       # the figures notebook looks for this name
DATASET = 'full'                      # -> data/full_benchmark.jsonl
PROMPT_VERSION = 'v2'
PRECISION = '4bit'                    # bfloat16 | float16 | float32 | 4bit | 8bit
BATCH_SIZE = 1
DEVICE = 'auto'
REPO_URL = ''                         # optional: git clone URL instead of an attached Kaggle Dataset

SESSION_LIMIT_HOURS = 11.0            # Kaggle GPU sessions stop at 12 h; a condition that would not fit
                                      # is skipped (and reported) so the finished results still get packaged.
RESUME = False                        # True: skip conditions whose metrics file already exists
PIN_SINGLE_GPU = True                 # use GPU 0 only; device_map='auto' would otherwise split a
                                      # small quantised model across both T4s and slow it down
MOCK = False                          # True: dependency-free smoke test (mock engine, no GPU/downloads)
MIN_TRANSFORMERS = '4.48.0'           # OLMo-2 / Llama-3.2 need a recent transformers
EXPECTED_RECORDS = 1850               # RQ1 1500 + RQ2 350 (warning only)
MIN_RECORDS = 1000                    # hard floor

MODE = 'cot' if USE_COT else 'no_cot'
MODELS_TO_CHECK = [MODEL]
CONDITIONS = [{'model': MODEL, 'cot': USE_COT, 'max_new_tokens': t} for t in TOKEN_BUDGETS]
print(f'{MODEL} | {MODE} | budgets={TOKEN_BUDGETS} | precision={PRECISION} | batch={BATCH_SIZE} | mock={MOCK}')


In [ ]:
# ---- Environment, repository, dependencies, pre-flight checks -----------------
import hashlib, json, os, shutil, subprocess, sys, urllib.error, urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

WORK_ROOT = Path(os.environ.get('DWS_WORK_ROOT', '/kaggle/working'))
INPUT_ROOT = Path(os.environ.get('DWS_INPUT_ROOT', '/kaggle/input'))
PROJECT_DIR = WORK_ROOT / 'StateMachine'
OUTPUT_ROOT = WORK_ROOT / (OUTPUT_NAME + ('_MOCK' if MOCK else ''))
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Environment for every child process (run_eval.py, pip, probes).
CHILD_ENV = os.environ.copy()
CHILD_ENV['PYTHONUNBUFFERED'] = '1'
CHILD_ENV['TOKENIZERS_PARALLELISM'] = 'false'
CHILD_ENV['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
if PIN_SINGLE_GPU and not MOCK:
    CHILD_ENV.setdefault('CUDA_VISIBLE_DEVICES', '0')


def stream(cmd, log_path=None, cwd=None, quiet_prefixes=('Processed',), throttle_s=60):
    """Run cmd and echo its output into the notebook (subprocess.run alone shows nothing in
    Jupyter), keep a full log, and return (returncode, last lines)."""
    log = open(log_path, 'w', encoding='utf-8') if log_path else None
    tail, last_quiet = [], 0.0
    proc = subprocess.Popen(cmd, cwd=cwd, env=CHILD_ENV, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            if log:
                log.write(line)
            tail.append(line)
            del tail[:-60]
            text = line.strip()
            if not text:
                continue
            if text.startswith(quiet_prefixes) or '%|' in text:   # progress bars: print once a minute
                if time.time() - last_quiet < throttle_s:
                    continue
                last_quiet = time.time()
            print(line, end='' if line.endswith('\n') else '\n', flush=True)
    finally:
        proc.stdout.close()
        rc = proc.wait()
        if log:
            log.close()
    return rc, ''.join(tail)


def pip_install(*requirements):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *requirements]
    result = subprocess.run(cmd, capture_output=True, text=True, env=CHILD_ENV)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError('pip install failed: ' + ' '.join(requirements))


# ---- Hugging Face token (Kaggle Secret HF_TOKEN) --------------------------------
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        _secret = UserSecretsClient().get_secret('HF_TOKEN')
        if _secret:
            os.environ['HF_TOKEN'] = _secret
            CHILD_ENV['HF_TOKEN'] = _secret
    except Exception:
        pass
HAS_HF_TOKEN = bool(os.environ.get('HF_TOKEN'))
print('HF_TOKEN:', 'found (value not printed)' if HAS_HF_TOKEN else 'not set')
if REQUIRES_HF_TOKEN and not HAS_HF_TOKEN and not MOCK:
    raise RuntimeError(
        f'{MODELS_TO_CHECK} includes a gated Hugging Face model. Add a Kaggle Secret named HF_TOKEN '
        '(Add-ons -> Secrets, attach it to this notebook) and accept the model licence '
        'on huggingface.co with the same account.')

# ---- Locate and copy the repository ---------------------------------------------
def is_repo(d):
    return (d / 'run_eval.py').exists() and (d / 'generate_all.py').exists() and (d / 'eval' / 'engine.py').exists()


def find_repo():
    here = Path.cwd().resolve()
    for d in [here, *list(here.parents)[:3]]:          # running from inside a checkout
        if is_repo(d):
            return [d]
    hits = []
    if INPUT_ROOT.exists():
        hits = [p.parent for p in INPUT_ROOT.rglob('run_eval.py') if is_repo(p.parent)]
    return sorted(hits, key=lambda d: (len(d.parts), str(d)))


def make_writable(root):
    for p in [root, *root.rglob('*')]:
        try:
            p.chmod(p.stat().st_mode | 0o700)
        except OSError:
            pass


if REPO_URL:
    os.chdir(WORK_ROOT)
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
    SOURCE = REPO_URL
else:
    hits = find_repo()
    if not hits:
        raise FileNotFoundError(
            'Repository not found. Attach the StateMachine repository as a Kaggle Dataset '
            f'(Add Input) so it appears under {INPUT_ROOT}, or set REPO_URL.')
    if len(hits) > 1:
        print('Several repository copies found, using the shallowest:', *hits, sep='\n  ')
    SOURCE = hits[0]
    if SOURCE.resolve() != PROJECT_DIR.resolve():
        os.chdir(WORK_ROOT)                              # never stand inside a dir we delete
        shutil.rmtree(PROJECT_DIR, ignore_errors=True)
        shutil.copytree(SOURCE, PROJECT_DIR, ignore=shutil.ignore_patterns(
            '.git', '__pycache__', '*.pyc', '.ipynb_checkpoints', 'results'))
        make_writable(PROJECT_DIR)                       # /kaggle/input copies can be read-only
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Source :', SOURCE)
print('Project:', PROJECT_DIR)

# ---- Validate the model keys against the repository registry ----------------------
REGISTRY = json.loads(subprocess.run(
    [sys.executable, '-c',
     'import json; from eval.models import CORE_MODELS as C, OPTIONAL_MODELS as O; '
     'print(json.dumps({k: [v.hf_model_id, v.revision] for k, v in {**C, **O}.items()}))'],
    cwd=PROJECT_DIR, env=CHILD_ENV, capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1])
unknown = [m for m in MODELS_TO_CHECK if m not in REGISTRY]
if unknown:
    raise KeyError(f'{unknown} not in eval/models.py: {sorted(REGISTRY)}')
for _m in MODELS_TO_CHECK:
    print(f'Model  : {_m} -> {REGISTRY[_m][0]} @ {REGISTRY[_m][1][:10]}')

# ---- Dependencies and GPU / network pre-flight (skipped for MOCK smoke tests) ------
ENV_INFO = {}
if not MOCK:
    pip_install('-r', str(PROJECT_DIR / 'requirements.txt'), f'transformers>={MIN_TRANSFORMERS}',
                'accelerate>=0.28.0', 'bitsandbytes>=0.43.0', 'pandas')
    probe = subprocess.run([sys.executable, '-c', """
import json, importlib
out = {}
for m in ('torch', 'transformers', 'bitsandbytes', 'accelerate', 'huggingface_hub'):
    try:
        out[m] = importlib.import_module(m).__version__
    except Exception as e:
        out[m] = 'MISSING ' + type(e).__name__
try:
    import torch
    out['cuda'] = bool(torch.cuda.is_available())
    out['gpus'] = [torch.cuda.get_device_name(i) + ' cc' + '.'.join(map(str, torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())]
except Exception:
    out['cuda'], out['gpus'] = False, []
print(json.dumps(out))
"""], env=CHILD_ENV, capture_output=True, text=True)
    ENV_INFO = json.loads(probe.stdout.strip().splitlines()[-1])
    print('Environment:', json.dumps(ENV_INFO, indent=2))
    from packaging.version import Version
    if ENV_INFO['transformers'].startswith('MISSING') or Version(ENV_INFO['transformers']) < Version(MIN_TRANSFORMERS):
        raise RuntimeError(f"transformers {ENV_INFO['transformers']} < {MIN_TRANSFORMERS}")
    if PRECISION in ('4bit', '8bit') and ENV_INFO['bitsandbytes'].startswith('MISSING'):
        raise RuntimeError('bitsandbytes is required for ' + PRECISION)
    if not ENV_INFO['cuda']:
        raise RuntimeError('No CUDA GPU visible. In Kaggle: Settings -> Accelerator -> GPU T4 x2 (or P100).')

    # Internet + model access (also validates the pinned revision and the licence for gated repos).
    try:
        urllib.request.urlopen('https://huggingface.co', timeout=20).close()
    except urllib.error.HTTPError:
        pass                                              # server answered -> network is on
    except Exception as exc:
        raise RuntimeError('huggingface.co is unreachable - switch Kaggle "Internet" ON in the '
                           'notebook settings (phone-verified accounts only).') from exc
    for _m in MODELS_TO_CHECK:
        hf_id, hf_rev = REGISTRY[_m]
        hub = subprocess.run([sys.executable, '-c', """
import os, sys
from huggingface_hub import hf_hub_download
try:
    print('OK', hf_hub_download(sys.argv[1], 'config.json', revision=sys.argv[2], token=os.environ.get('HF_TOKEN') or None))
except Exception as e:
    print('ERR', type(e).__name__, str(e)[:600]); sys.exit(2)
""", hf_id, hf_rev], env=CHILD_ENV, capture_output=True, text=True)
        print(f'{_m}:', hub.stdout.strip().splitlines()[-1] if hub.stdout.strip() else hub.stderr[-500:])
        if hub.returncode != 0:
            raise RuntimeError(f'Cannot access {hf_id}@{hf_rev}. For Llama: accept the licence on '
                               'huggingface.co with the account that owns HF_TOKEN.')


In [ ]:
# ---- Benchmark dataset: use the uploaded one, otherwise generate; then validate --
dataset_path = PROJECT_DIR / 'data' / 'full_benchmark.jsonl'
if dataset_path.exists():
    print('Using dataset shipped with the repository upload:', dataset_path)
else:
    print('No dataset in the upload - generating (python generate_all.py --write)...')
    rc, tail = stream([sys.executable, 'generate_all.py', '--write'], cwd=PROJECT_DIR)
    if rc != 0:
        raise RuntimeError('generate_all.py failed (see output above).')
    if not dataset_path.exists():
        raise FileNotFoundError(dataset_path)

records = [json.loads(line) for line in dataset_path.open(encoding='utf-8') if line.strip()]
for field in ('instance_id', 'context', 'question', 'gold_answer'):
    bad = [r.get('instance_id') for r in records if field not in r]
    if bad:
        raise ValueError(f'{len(bad)} records lack {field!r}, e.g. {bad[:3]}')
ids = [r['instance_id'] for r in records]
if len(ids) != len(set(ids)):
    raise ValueError('Duplicate instance_id values in the dataset.')
RECORD_COUNT = len(records)
by_experiment = Counter(r.get('experiment') for r in records)
by_family = Counter((r.get('experiment'), r.get('family')) for r in records)
print('records:', RECORD_COUNT, '| by experiment:', dict(by_experiment))
if RECORD_COUNT < MIN_RECORDS:
    raise ValueError(f'Unexpected benchmark size: {RECORD_COUNT} (< {MIN_RECORDS}).')
for tag in ('rq1_mutation_depth', 'rq2_interference'):
    if not by_experiment.get(tag):
        raise ValueError(f'No {tag} records in the dataset.')
if RECORD_COUNT != EXPECTED_RECORDS:
    print(f'WARNING: expected {EXPECTED_RECORDS} records, found {RECORD_COUNT}. '
          'Make sure every notebook/account runs the SAME dataset (compare the sha256 below).')
DATASET_SHA256 = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
print('sha256 :', DATASET_SHA256)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
_ = (OUTPUT_ROOT / 'run_manifest.json').write_text(json.dumps({
    'models': {m: {'hf_model_id': REGISTRY[m][0], 'revision': REGISTRY[m][1]} for m in MODELS_TO_CHECK},
    'conditions': CONDITIONS, 'precision': PRECISION, 'batch_size': BATCH_SIZE,
    'prompt_version': PROMPT_VERSION, 'mock': MOCK, 'dataset': dataset_path.name,
    'dataset_sha256': DATASET_SHA256, 'dataset_records': RECORD_COUNT,
    'records_by_experiment': {str(k): v for k, v in by_experiment.items()},
    'records_by_experiment_family': {f'{k[0]}/{k[1]}': v for k, v in by_family.items()},
    'environment': ENV_INFO, 'source': str(SOURCE),
}, indent=2), encoding='utf-8')


In [ ]:
# ---- Helpers: per-condition paths, summary table, packaging ---------------------
DATASET_STEM = dataset_path.stem                     # run_eval.py names artifacts after it


def mode_of(c):
    return 'cot' if c['cot'] else 'no_cot'


def condition_name(c):
    return f"{c['model']}_{mode_of(c)}_{c['max_new_tokens']}"


def metrics_file(c):
    # run_eval.py: <output-dir>/<model>/<cot|no_cot>_<max_new_tokens>/<dataset>_metrics.json
    return (OUTPUT_ROOT / condition_name(c) / c['model'] / f"{mode_of(c)}_{c['max_new_tokens']}"
            / f'{DATASET_STEM}_metrics.json')


def build_summary():
    rows = []
    for c in CONDITIONS:
        path = metrics_file(c)
        if not path.exists():
            continue
        m = json.loads(path.read_text(encoding='utf-8'))
        rows.append({
            'model': c['model'], 'cot': c['cot'], 'max_new_tokens': c['max_new_tokens'],
            'accuracy': m['overall_accuracy'], 'prompt_version': m.get('prompt_version'),
            'rq1_onset_L_T': m.get('rq1_failure_onset_L_T'),
            'rq3_onset_L_D': m.get('rq3_failure_onset_L_D'),
            'stepwise_accuracy': m.get('stepwise_accuracy'),
            'missing_predictions': m.get('missing_predictions'),
            'instances': m['total_instances'], 'runtime_minutes': m['elapsed_seconds'] / 60,
        })
    return pd.DataFrame(rows)


def package_results():
    """(Re)write run_status/summary CSVs and the download archive. Safe to call repeatedly."""
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(run_status).to_csv(OUTPUT_ROOT / 'run_status.csv', index=False)
    summary = build_summary()
    summary.to_csv(OUTPUT_ROOT / 'corrected_condition_summary.csv', index=False)
    archive = shutil.make_archive(str(WORK_ROOT / OUTPUT_ROOT.name), 'zip',
                                  root_dir=WORK_ROOT, base_dir=OUTPUT_ROOT.name)
    return summary, archive


run_status = []


In [ ]:
# ---- Run every condition as an independent run_eval.py process --------------------
durations = {}                                      # (model, cot) -> minutes of completed runs
for index, c in enumerate(CONDITIONS, start=1):
    name, tokens = condition_name(c), c['max_new_tokens']
    output_dir = OUTPUT_ROOT / name
    entry = {'condition': name, 'model': c['model'], 'cot': c['cot'], 'max_new_tokens': tokens,
             'return_code': None, 'minutes': 0.0, 'status': ''}

    if RESUME and metrics_file(c).exists():
        entry.update(return_code=0, status='cached')
        run_status.append(entry)
        print(f'[{index}/{len(CONDITIONS)}] {name}: metrics already present - skipped (RESUME=True)')
        continue

    elapsed_h = (time.time() - NB_START) / 3600
    previous = durations.get((c['model'], c['cot']))
    if previous and elapsed_h + 1.1 * max(previous) / 60 > SESSION_LIMIT_HOURS:
        entry.update(status='skipped_time_limit')
        run_status.append(entry)
        print(f'[{index}/{len(CONDITIONS)}] {name}: SKIPPED - {elapsed_h:.1f} h used, the previous comparable '
              f'run took {max(previous):.0f} min, limit {SESSION_LIMIT_HOURS} h.')
        continue

    command = [sys.executable, 'run_eval.py', '--model', c['model'], '--dataset', DATASET,
               '--device', DEVICE, '--precision', PRECISION, '--batch-size', str(BATCH_SIZE),
               '--max-new-tokens', str(tokens), '--output-dir', str(output_dir),
               '--prompt-version', PROMPT_VERSION, '--overwrite']
    if c['cot']:
        command.append('--cot')
    if MOCK:
        command.append('--mock')

    print(f'\n=== [{index}/{len(CONDITIONS)}] {name} started {time.strftime("%H:%M:%S")} ===')
    output_dir.mkdir(parents=True, exist_ok=True)
    started = time.time()
    rc, tail = stream(command, log_path=output_dir / 'run.log', cwd=PROJECT_DIR)
    minutes = (time.time() - started) / 60
    entry.update(return_code=rc, minutes=round(minutes, 2),
                 status='ok' if rc == 0 and metrics_file(c).exists() else 'failed')
    if entry['status'] == 'failed':
        print(f'[{name}] FAILED (return code {rc}); last output:\n{tail[-3000:]}')
    else:
        durations.setdefault((c['model'], c['cot']), []).append(minutes)
    run_status.append(entry)
    package_results()                      # keep a downloadable archive up to date after every condition

display(pd.DataFrame(run_status))


In [ ]:
# ---- Summary ----------------------------------------------------------------------
summary, archive_path = package_results()
display(summary)
for _, row in summary.iterrows():
    tag = f"{row['model']} {'cot' if row['cot'] else 'no_cot'} {row['max_new_tokens']}"
    if int(row['instances']) != RECORD_COUNT:
        print(f"WARNING: {tag} evaluated {int(row['instances'])} of {RECORD_COUNT} records")
    if pd.notna(row['missing_predictions']) and row['missing_predictions'] > 0:
        print(f"WARNING: {tag} produced {int(row['missing_predictions'])} empty predictions")
print('Saved results to', OUTPUT_ROOT)


In [ ]:
# ---- Archive for download ------------------------------------------------------------
summary, archive_path = package_results()           # also picks up files written by the analysis cell
print('Created archive:', archive_path, f'({Path(archive_path).stat().st_size / 1e6:.1f} MB)')
problems = [s for s in run_status if s['status'] not in ('ok', 'cached')]
if problems:
    print('\nNOT COMPLETED:', *[f"  {s['condition']}: {s['status']}" for s in problems], sep='\n')
    if any(s['status'] == 'failed' for s in problems):
        raise RuntimeError('At least one run failed - see its log above / run.log in the archive.')
else:
    print('All conditions completed.')
